# FaceXplain — recompute the original paper's analyses on the clean split

This notebook refreshes the numbers of the **original paper sections** so the text can stay almost unchanged. It uses the clean split and models created by `FaceXplain_clean_rerun` (same Drive folder).

| Part | Original section it refreshes | Time (T4) |
|---|---|---|
| A | Model refinement (class-weighted VGG16, enhanced augmentation), 3 seeds, vs. baseline VGG16 | ~25 min |
| B | Grad-CAM vs Grad-CAM++ eye-region activation score (ERAS) with MediaPipe | ~5 min |
| C | LIME: Jaccard stability, fidelity, comprehensiveness, sufficiency | ~8 min |
| D | MediaPipe landmark measurements (Mann–Whitney U) on the de-duplicated dataset | ~5 min |
| E | ViT-B/16 comparator on the same split + attention maps | ~15 min |

**If Colab disconnects:** just *Runtime → Run all* again — finished parts are loaded from Drive, not recomputed.

In [ ]:
OUT_DIR  = "/content/drive/MyDrive/FaceXplain_clean_rerun"   # same folder as the clean run
SEEDS    = [42, 123, 2024]
XAI_SEED = 42
N_XAI    = 30          # correctly classified ASD test images for Grad-CAM/Grad-CAM++/LIME (as in the paper)
LIME_SAMPLES = 1500
RUN_VIT  = True
IMG_SIZE, BATCH_SIZE = 224, 32
CLASS_WEIGHT_ASD = 2.0    # refined model: w_ASD = 2.0, w_NonASD = 1.0 (as in the paper)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q lime mediapipe timm
!wget -q -nc -O /content/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

In [ ]:
import os, json, random, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
for g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError: pass   # leave GPU memory for the ViT (PyTorch) part
from tensorflow import keras
from tensorflow.keras import layers
from scipy import stats
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve)

FIG = os.path.join(OUT_DIR, "figures_original_sections"); os.makedirs(FIG, exist_ok=True)
RES = os.path.join(OUT_DIR, "original_sections");         os.makedirs(RES, exist_ok=True)
W_DIR = os.path.join(OUT_DIR, "weights")

split = pd.read_csv(os.path.join(OUT_DIR, "final_split.csv"))
S = {k: (g.path.values, g.label.values) for k, g in split.groupby("split")}
tr_p, tr_y = S["train"]; va_p, va_y = S["val"]; te_p, te_y = S["test"]
print("train/val/test:", len(tr_p), len(va_p), len(te_p))

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)

def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p); return float(np.clip(thr[np.argmax(tpr - fpr)], 1e-6, 1 - 1e-6))

def metrics_at(y, p, thr):
    yp = (p >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0, 1]).ravel()
    return {"threshold": thr, "accuracy": accuracy_score(y, yp), "precision": precision_score(y, yp, zero_division=0),
            "recall_sensitivity": recall_score(y, yp, zero_division=0), "specificity": tn / (tn + fp),
            "f1": f1_score(y, yp, zero_division=0), "auc": roc_auc_score(y, p),
            "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)}

def bootstrap_ci(y, p, thr, n=2000, seed=0):
    rng = np.random.default_rng(seed); a, u = [], []
    for _ in range(n):
        s = rng.integers(0, len(y), len(y))
        if len(np.unique(y[s])) < 2: continue
        a.append(accuracy_score(y[s], (p[s] >= thr).astype(int))); u.append(roc_auc_score(y[s], p[s]))
    return np.percentile(a, [2.5, 97.5]).tolist(), np.percentile(u, [2.5, 97.5]).tolist()

def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1; i = j
    out = np.empty(N); out[J] = T; return out

def delong_p(y, p1, p2):
    pos = y == 1; m, n = pos.sum(), (~pos).sum(); P = np.vstack([p1, p2])
    tx = np.array([_midrank(r[pos]) for r in P]); ty = np.array([_midrank(r[~pos]) for r in P])
    tz = np.array([_midrank(np.concatenate([r[pos], r[~pos]])) for r in P])
    auc = tz[:, :m].sum(1) / m / n - (m + 1) / (2 * n)
    v01 = (tz[:, :m] - tx) / n; v10 = 1 - (tz[:, m:] - ty) / m
    Sg = np.cov(v01) / m + np.cov(v10) / n; l = np.array([1, -1])
    z = (auc[0] - auc[1]) / np.sqrt(l @ Sg @ l); return float(2 * stats.norm.sf(abs(z)))

AUTOTUNE = tf.data.AUTOTUNE
def load_img(path, label, w):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.cast(tf.image.resize(img, [IMG_SIZE, IMG_SIZE]), tf.float32), tf.cast(label, tf.float32), w
def make_ds(paths, labels, shuffle, seed, weights=None):
    w = np.ones(len(paths), np.float32) if weights is None else weights.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, w)).map(load_img, num_parallel_calls=AUTOTUNE)
    if shuffle: ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
def predict_probs(model, paths):
    ds = make_ds(paths, np.zeros(len(paths)), False, 0).map(lambda x, y, w: x)
    return model.predict(ds, verbose=0).ravel()
def load_raw(path):
    return np.asarray(Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32)

## Part A — Refined VGG16 (class-weighted, enhanced augmentation), 3 seeds

In [ ]:
class SkinToneJitter(layers.Layer):
    # skin-tone variation: small random hue/saturation shifts, training only
    def call(self, x, training=None):
        if not training:
            return x
        y = x / 255.0
        y = tf.image.random_hue(y, 0.03)
        y = tf.image.random_saturation(y, 0.85, 1.15)
        return tf.clip_by_value(y, 0.0, 1.0) * 255.0

def build_vgg16(refined):
    base = keras.applications.VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    aug_layers = [layers.RandomFlip("horizontal"), layers.RandomRotation(0.05), layers.RandomZoom(0.1),
                  layers.RandomContrast(0.1), layers.RandomBrightness(0.1)]
    if refined:   # enhanced augmentation: + translation and skin-tone variation
        aug_layers += [layers.RandomTranslation(0.08, 0.08), SkinToneJitter()]
    aug = keras.Sequential(aug_layers, name="augmentation")
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = keras.applications.vgg16.preprocess_input(aug(inp))
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="head_bn")(x)
    x = layers.Dense(128, activation="relu", name="head_dense")(x)
    x = layers.Dropout(0.5, name="head_dropout")(x)
    out = layers.Dense(1, activation="sigmoid", name="head_out")(x)
    return keras.Model(inp, out), base

def unfreeze_block5(base):
    base.trainable = True
    for l in base.layers: l.trainable = l.name.startswith("block5")

def compile_m(m, lr):
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
              metrics=["accuracy", keras.metrics.AUC(name="auc")], weighted_metrics=[])

def run_refined(seed):
    tag = f"VGG16refined_seed{seed}"; wp = os.path.join(W_DIR, tag + ".weights.h5"); rp = os.path.join(RES, tag + ".json")
    if os.path.exists(wp) and os.path.exists(rp):
        m, b = build_vgg16(True); unfreeze_block5(b); m.load_weights(wp)
        saved = json.load(open(rp)); print(f">>> {tag}: loaded"); return m, b, saved["row"], saved["hist"]
    set_seed(seed); m, b = build_vgg16(True)
    w = np.where(tr_y == 1, CLASS_WEIGHT_ASD, 1.0)
    tr_ds = make_ds(tr_p, tr_y, True, seed, w); va_ds = make_ds(va_p, va_y, False, seed)
    es = lambda: keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True)
    compile_m(m, 1e-4); h1 = m.fit(tr_ds, validation_data=va_ds, epochs=15, callbacks=[es()], verbose=2)
    unfreeze_block5(b); compile_m(m, 1e-5); h2 = m.fit(tr_ds, validation_data=va_ds, epochs=15, callbacks=[es()], verbose=2)
    pv = predict_probs(m, va_p); thr = youden_threshold(va_y, pv); pt = predict_probs(m, te_p)
    mm = metrics_at(te_y, pt, thr); (al, ah), (ul, uh) = bootstrap_ci(te_y, pt, thr, seed=seed)
    row = {"model": "VGG16-refined", "seed": seed, **mm, "acc_CI95_low": al, "acc_CI95_high": ah,
           "auc_CI95_low": ul, "auc_CI95_high": uh, "epochs_stage1": len(h1.history["loss"]), "epochs_stage2": len(h2.history["loss"])}
    row = {k: (v.item() if hasattr(v, "item") else v) for k, v in row.items()}
    hist = {k: [float(v) for v in h1.history[k] + h2.history[k]] for k in h1.history}
    m.save_weights(wp); json.dump({"row": row, "hist": hist}, open(rp, "w"), indent=1)
    pd.DataFrame({"path": te_p, "y_true": te_y, "p_asd": pt, "y_pred": (pt >= thr).astype(int)}).to_csv(
        os.path.join(RES, f"test_predictions_{tag}.csv"), index=False)
    print(f">>> {tag}: acc {mm['accuracy']:.4f} sens {mm['recall_sensitivity']:.4f} AUC {mm['auc']:.4f}")
    return m, b, row, hist

ref_models, ref_rows, ref_hist = {}, [], {}
for s in SEEDS:
    m, b, row, hist = run_refined(s); ref_models[s] = (m, b); ref_rows.append(row); ref_hist[s] = hist
ref_df = pd.DataFrame(ref_rows)

In [ ]:
# Baseline VGG16 (from the clean run) vs refined VGG16 — same split, same seeds, same test set
base_rows = [json.load(open(os.path.join(OUT_DIR, "runs", f"VGG16_seed{s}.json")))["row"] for s in SEEDS]
base_df = pd.DataFrame(base_rows)
cols = ["accuracy", "precision", "recall_sensitivity", "specificity", "f1", "auc"]
comp = pd.DataFrame({"Baseline VGG16": base_df[cols].mean(), "Baseline SD": base_df[cols].std(),
                     "Refined VGG16": ref_df[cols].mean(), "Refined SD": ref_df[cols].std()})
print((comp * 100).round(2))

pvals = []
for s in SEEDS:
    b = pd.read_csv(os.path.join(OUT_DIR, f"test_predictions_VGG16_seed{s}.csv")).sort_values("path")
    r = pd.read_csv(os.path.join(RES, f"test_predictions_VGG16refined_seed{s}.csv")).sort_values("path")
    assert (b.path.values == r.path.values).all()
    pvals.append(delong_p(b.y_true.values, r.p_asd.values, b.p_asd.values))
print("DeLong p (refined vs baseline, per seed):", np.round(pvals, 4))

# confusion matrices: class-specific results (as in the paper's class-specific table)
for name, d in [("Baseline", base_df), ("Refined", ref_df)]:
    print(name, "per seed TP/FN/TN/FP:", d[["seed", "TP", "FN", "TN", "FP"]].values.tolist())

comp.to_csv(os.path.join(RES, "baseline_vs_refined.csv"))
ref_df.to_csv(os.path.join(RES, "refined_per_seed.csv"), index=False)
json.dump({"delong_p_refined_vs_baseline": pvals}, open(os.path.join(RES, "refined_delong.json"), "w"))

# figures: refined confusion matrix + ROC (seed 42) and training curves
m42 = ref_models[XAI_SEED][0]; r42 = ref_df[ref_df.seed == XAI_SEED].iloc[0]; p42 = predict_probs(m42, te_p)
cm = confusion_matrix(te_y, (p42 >= r42.threshold).astype(int), labels=[0, 1])
fig, ax = plt.subplots(1, 2, figsize=(10, 4)); ax[0].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm): ax[0].text(j, i, v, ha="center", va="center", fontsize=14)
ax[0].set_xticks([0, 1]); ax[0].set_yticks([0, 1]); ax[0].set_xticklabels(["Non-ASD", "ASD"]); ax[0].set_yticklabels(["Non-ASD", "ASD"])
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True"); ax[0].set_title("Refined VGG16: test confusion matrix")
fpr, tpr, _ = roc_curve(te_y, p42); ax[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(te_y, p42):.3f}"); ax[1].plot([0, 1], [0, 1], "--", c="grey")
ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate"); ax[1].legend(); ax[1].set_title("Refined VGG16: test ROC")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "refined_confusion_roc.png"), dpi=200); plt.show()
h = ref_hist[XAI_SEED]; fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h["accuracy"], label="train"); ax[0].plot(h["val_accuracy"], label="validation"); ax[0].set_title("Accuracy")
ax[1].plot(h["loss"], label="train"); ax[1].plot(h["val_loss"], label="validation"); ax[1].set_title("Loss")
for a in ax: a.axvline(int(r42.epochs_stage1) - 0.5, ls="--", c="grey"); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(FIG, "refined_training_curves.png"), dpi=200); plt.show()

## Part B — Grad-CAM vs Grad-CAM++ eye-region activation score (ERAS)

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
landmarker = mp_vision.FaceLandmarker.create_from_options(
    mp_vision.FaceLandmarkerOptions(base_options=mp_python.BaseOptions(model_asset_path="/content/face_landmarker.task"),
                                    num_faces=1))

EYE_A = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
EYE_B = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]

def landmarks_px(img_uint8):
    res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(img_uint8)))
    if not res.face_landmarks: return None
    h, w = img_uint8.shape[:2]
    return np.array([[p.x * w, p.y * h] for p in res.face_landmarks[0]])

def eye_mask(lm, h=IMG_SIZE, w=IMG_SIZE):
    pts = lm[EYE_A + EYE_B]
    x0, y0 = pts.min(0); x1, y1 = pts.max(0)
    x0 -= 0.05 * w; x1 += 0.05 * w; y0 -= 0.04 * h; y1 += 0.04 * h      # 5% horizontal, 4% vertical padding
    m = np.zeros((h, w), bool)
    m[max(int(y0), 0):min(int(np.ceil(y1)), h), max(int(x0), 0):min(int(np.ceil(x1)), w)] = True
    return m

xai_model, xai_base = ref_models[XAI_SEED]
conv_model = keras.Model(xai_base.inputs, [xai_base.get_layer("block5_conv3").output, xai_base.output])
head = [xai_model.get_layer(n) for n in ["gap", "head_bn", "head_dense", "head_dropout", "head_out"]]
prep = keras.applications.vgg16.preprocess_input

def cams(img_raw):
    # Grad-CAM and Grad-CAM++ w.r.t. the ASD logit, from block5_conv3
    x = prep(tf.convert_to_tensor(img_raw[None].copy()))
    with tf.GradientTape() as tape:
        conv, feat = conv_model(x, training=False); hh = feat
        for l in head: hh = l(hh, training=False)
        p = tf.clip_by_value(hh[:, 0], 1e-7, 1 - 1e-7); logit = tf.math.log(p / (1 - p))
    g = tape.gradient(logit, conv)[0].numpy(); A = conv[0].numpy()
    w_gc = g.mean(axis=(0, 1)); cam_gc = np.maximum((A * w_gc).sum(-1), 0)
    g2, g3 = g ** 2, g ** 3; denom = 2 * g2 + A.sum(axis=(0, 1))[None, None, :] * g3
    alpha = g2 / np.where(denom != 0, denom, 1.0)
    w_pp = (alpha * np.maximum(g, 0)).sum(axis=(0, 1)); cam_pp = np.maximum((A * w_pp).sum(-1), 0)
    up = lambda c: np.asarray(Image.fromarray((c / (c.max() + 1e-8) * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0
    return up(cam_gc), up(cam_pp), float(p[0])

thr42 = float(ref_df[ref_df.seed == XAI_SEED].threshold.iloc[0])
p_ref = predict_probs(xai_model, te_p)
cand = np.where((te_y == 1) & (p_ref >= thr42))[0]
rng = np.random.default_rng(XAI_SEED); rng.shuffle(cand)

eras_path = os.path.join(RES, "eras.csv")
rows, chosen, store = [], [], {}
for i in cand:
    if len(chosen) == N_XAI: break
    img = load_raw(te_p[i]); lm = landmarks_px(img.astype(np.uint8))
    if lm is None: continue
    M = eye_mask(lm); gc, pp, p = cams(img)
    rows.append({"path": te_p[i], "p_asd": p, "eras_gradcam": (gc * M).sum() / (gc.sum() + 1e-8),
                 "eras_gradcampp": (pp * M).sum() / (pp.sum() + 1e-8), "uniform_baseline": M.mean()})
    chosen.append(i); store[i] = (img, M, gc, pp, lm)
eras = pd.DataFrame(rows); eras.to_csv(eras_path, index=False)
w = stats.wilcoxon(eras.eras_gradcampp, eras.eras_gradcam)
print(f"Images: {len(eras)} correctly classified ASD test images with a detected face")
print(eras[["eras_gradcam", "eras_gradcampp", "uniform_baseline"]].agg(["mean", "std"]).round(4))
print(f"Wilcoxon signed-rank (Grad-CAM++ vs Grad-CAM): p = {w.pvalue:.2e}")
for c in ["eras_gradcam", "eras_gradcampp"]:
    t = stats.wilcoxon(eras[c], eras.uniform_baseline); print(f"{c} vs uniform baseline: p = {t.pvalue:.2e}")

fig, ax = plt.subplots(figsize=(5, 3.5))
mu = [eras.eras_gradcam.mean(), eras.eras_gradcampp.mean()]; sd = [eras.eras_gradcam.std(), eras.eras_gradcampp.std()]
ax.bar(["Grad-CAM", "Grad-CAM++"], mu, yerr=sd, capsize=5, color=["#3b6ea5", "#e3923b"])
ax.axhline(eras.uniform_baseline.mean(), ls="--", c="grey", label="uniform-attention baseline"); ax.legend()
ax.set_ylabel("Eye-region activation score"); plt.tight_layout(); plt.savefig(os.path.join(FIG, "eras_comparison.png"), dpi=200); plt.show()

show = chosen[:4]; fig, ax = plt.subplots(len(show), 3, figsize=(9, 3 * len(show)))
for r, i in enumerate(show):
    img, M, gc, pp, lm = store[i]
    ax[r, 0].imshow(img.astype(np.uint8)); ax[r, 0].contour(M, colors="lime", linewidths=1); ax[r, 0].set_title("Image + eye region", fontsize=9)
    ax[r, 1].imshow(img.astype(np.uint8)); ax[r, 1].imshow(gc, cmap="jet", alpha=.45); ax[r, 1].set_title("Grad-CAM", fontsize=9)
    ax[r, 2].imshow(img.astype(np.uint8)); ax[r, 2].imshow(pp, cmap="jet", alpha=.45); ax[r, 2].set_title("Grad-CAM++", fontsize=9)
    for a in ax[r]: a.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "gradcampp_examples.png"), dpi=200); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(np.mean([store[i][2] for i in chosen], 0), cmap="jet"); ax[0].set_title(f"Average Grad-CAM (n={len(chosen)})")
ax[1].imshow(np.mean([store[i][3] for i in chosen], 0), cmap="jet"); ax[1].set_title(f"Average Grad-CAM++ (n={len(chosen)})")
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "average_gradcam_gradcampp.png"), dpi=200); plt.show()

## Part C — LIME reliability (Jaccard stability, fidelity, comprehensiveness, sufficiency)

In [ ]:
from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
seg_fn = SegmentationAlgorithm("quickshift", kernel_size=4, max_dist=200, ratio=0.2, random_seed=0)
GREY = np.array([124, 116, 104], np.float32)
def f_asd(batch): return xai_model.predict(np.asarray(batch, np.float32), batch_size=64, verbose=0).ravel()
def classifier_fn(batch): p = f_asd(batch); return np.stack([1 - p, p], 1)

lime_path = os.path.join(RES, "lime_metrics.csv")
if os.path.exists(lime_path):
    lime_df = pd.read_csv(lime_path); print("LIME metrics loaded")
else:
    rows = []
    for k, i in enumerate(chosen):
        img = store[i][0]; M = store[i][1]
        runs = []
        for rs in (0, 1):
            exp = lime_image.LimeImageExplainer(random_state=rs).explain_instance(
                img.astype(np.double), classifier_fn, labels=(1,), top_labels=None, hide_color=None,
                num_samples=LIME_SAMPLES, batch_size=100, segmentation_fn=seg_fn, random_seed=rs)
            runs.append(exp)
        segs = runs[0].segments; n_seg = len(np.unique(segs))
        top = lambda e, k: [s for s, wt in sorted(e.local_exp[1], key=lambda t: -t[1])[:k] if wt > 0]
        A, B = set(top(runs[0], 10)), set(top(runs[1], 10))
        R = top(runs[0], 5); Rmask = np.isin(segs, R)
        p0 = f_asd(img[None])[0]
        p_removed = f_asd(np.where(Rmask[..., None], GREY, img)[None])[0]
        p_only = f_asd(np.where(Rmask[..., None], img, GREY)[None])[0]
        rows.append({"path": te_p[i], "n_superpixels": n_seg, "jaccard": len(A & B) / max(len(A | B), 1),
                     "fidelity_R2": runs[0].score, "comprehensiveness": p0 - p_removed, "sufficiency": p0 - p_only,
                     "top5_periocular_overlap": (Rmask & M).sum() / max(Rmask.sum(), 1)})
        if k == 0: lime_example = (img, M, Rmask)
        print(f"LIME {k+1}/{len(chosen)}", end="\r")
    lime_df = pd.DataFrame(rows); lime_df.to_csv(lime_path, index=False)
print(lime_df.drop(columns="path").agg(["mean", "std"]).round(4))

if "lime_example" in globals():
    img, M, Rmask = lime_example
    fig, ax = plt.subplots(1, 3, figsize=(10, 3.6))
    ax[0].imshow(img.astype(np.uint8)); ax[0].contour(M, colors="lime"); ax[0].set_title("Image + eye region")
    ax[1].imshow(np.where(Rmask[..., None], img, 255).astype(np.uint8)); ax[1].set_title("Top-5 LIME superpixels")
    ax[2].imshow(img.astype(np.uint8)); ax[2].imshow(np.ma.masked_where(~Rmask, Rmask), cmap="autumn", alpha=.5)
    ax[2].contour(M, colors="lime"); ax[2].set_title("Overlap with eye region")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.savefig(os.path.join(FIG, "lime_example.png"), dpi=200); plt.show()

## Part D — MediaPipe landmark measurements (de-duplicated dataset, all splits)

In [ ]:
lm_path = os.path.join(RES, "landmark_measurements.csv")
def D(lm, a, b): return float(np.linalg.norm(lm[a] - lm[b]))
if os.path.exists(lm_path):
    lmd = pd.read_csv(lm_path); print("landmarks loaded")
else:
    rows = []
    for k, (p, y) in enumerate(zip(split.path.values, split.label.values)):
        img = np.asarray(Image.open(p).convert("RGB")); lm = landmarks_px(img)
        if lm is None: continue
        ie = D(lm, 133, 362)                                  # inter-eye (inner canthi) distance for normalization
        ecA = (lm[33] + lm[133]) / 2; ecB = (lm[362] + lm[263]) / 2
        rows.append({"path": p, "label": y,
            "Left Eye Openness": D(lm, 159, 145) / ie, "Right Eye Openness": D(lm, 386, 374) / ie,
            "Brow Height Left": float(np.linalg.norm(lm[105] - ecA)) / ie, "Brow Height Right": float(np.linalg.norm(lm[334] - ecB)) / ie,
            "Mouth Width": D(lm, 61, 291) / ie, "Mouth Openness": D(lm, 13, 14) / ie, "Nose-to-Mouth Distance": D(lm, 1, 13) / ie})
        if k % 200 == 0: print(f"{k}/{len(split)}", end="\r")
    lmd = pd.DataFrame(rows); lmd.to_csv(lm_path, index=False)
feats = [c for c in lmd.columns if c not in ("path", "label")]
out = []
for f in feats:
    a, b = lmd[lmd.label == 1][f], lmd[lmd.label == 0][f]; u = stats.mannwhitneyu(a, b, alternative="two-sided")
    out.append({"Feature": f, "ASD Mean": a.mean(), "Non-ASD Mean": b.mean(), "p": u.pvalue,
                "rank_biserial": 1 - 2 * u.statistic / (len(a) * len(b))})
lt = pd.DataFrame(out).sort_values("p")
m_ = len(lt); adj = np.minimum(1, np.maximum.accumulate((m_ - np.arange(m_)) * lt.p.values)); lt["p_holm"] = adj
lt["Significant (Holm)"] = np.where(lt.p_holm < 0.05, "Yes", "No"); lt = lt.sort_index()
print(f"Faces detected: {len(lmd)} of {len(split)} images (ASD {int((lmd.label==1).sum())}, Non-ASD {int((lmd.label==0).sum())})")
print(lt.round(4).to_string(index=False)); lt.to_csv(os.path.join(RES, "landmark_stats.csv"), index=False)
fig, ax = plt.subplots(figsize=(9, 3.8)); x = np.arange(len(feats))
ax.bar(x - 0.2, lt["ASD Mean"], 0.4, label="ASD"); ax.bar(x + 0.2, lt["Non-ASD Mean"], 0.4, label="Non-ASD")
ax.set_xticks(x); ax.set_xticklabels(feats, rotation=25, ha="right"); ax.set_ylabel("Normalized measurement"); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(FIG, "landmark_comparison.png"), dpi=200); plt.show()

## Part E — ViT-B/16 comparator on the same split (PyTorch / timm)

In [ ]:
if RUN_VIT:
    import torch, timm, torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    import torchvision.transforms as T
    keras.backend.clear_session()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    vit_w = os.path.join(W_DIR, "ViTB16_seed42.pt"); vit_r = os.path.join(RES, "vit_result.json")
    norm = T.Normalize([0.5] * 3, [0.5] * 3)
    tf_train = T.Compose([T.Resize((224, 224)), T.RandomHorizontalFlip(), T.RandomRotation(10),
                          T.ColorJitter(0.1, 0.1, 0.1, 0.02), T.ToTensor(), norm])
    tf_eval = T.Compose([T.Resize((224, 224)), T.ToTensor(), norm])
    class DS(Dataset):
        def __init__(s, p, y, t): s.p, s.y, s.t = p, y, t
        def __len__(s): return len(s.p)
        def __getitem__(s, i): return s.t(Image.open(s.p[i]).convert("RGB")), torch.tensor(float(s.y[i]))
    def vit_predict(model, paths):
        dl = DataLoader(DS(paths, np.zeros(len(paths)), tf_eval), batch_size=64, num_workers=2); out = []
        model.eval()
        with torch.no_grad(), torch.autocast(dev, enabled=dev == "cuda"):
            for x, _ in dl: out.append(torch.sigmoid(model(x.to(dev)).float().squeeze(1)).cpu().numpy())
        return np.concatenate(out)
    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    vit = timm.create_model("vit_base_patch16_224.augreg_in21k_ft_in1k", pretrained=True, num_classes=1).to(dev)
    if os.path.exists(vit_w) and os.path.exists(vit_r):
        vit.load_state_dict(torch.load(vit_w, map_location=dev)); vit_row = json.load(open(vit_r)); print("ViT loaded")
    else:
        dl = DataLoader(DS(tr_p, tr_y, tf_train), batch_size=32, shuffle=True, num_workers=2, drop_last=True)
        opt = torch.optim.AdamW(vit.parameters(), lr=2e-5, weight_decay=0.05)
        scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda"); lossf = nn.BCEWithLogitsLoss()
        best, best_state, bad = -1, None, 0
        for ep in range(15):
            vit.train()
            for x, y in dl:
                opt.zero_grad()
                with torch.autocast(dev, enabled=dev == "cuda"): loss = lossf(vit(x.to(dev)).squeeze(1), y.to(dev))
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            va = roc_auc_score(va_y, vit_predict(vit, va_p)); print(f"ViT epoch {ep+1}: val AUC {va:.4f}")
            if va > best: best, bad, best_state = va, 0, {k: v.detach().cpu().clone() for k, v in vit.state_dict().items()}
            else:
                bad += 1
                if bad >= 4: break
        vit.load_state_dict(best_state); torch.save(best_state, vit_w)
        thr = youden_threshold(va_y, vit_predict(vit, va_p)); pt = vit_predict(vit, te_p)
        mm = metrics_at(te_y, pt, thr); (al, ah), (ul, uh) = bootstrap_ci(te_y, pt, thr)
        vit_row = {k: (v.item() if hasattr(v, "item") else v) for k, v in
                   {**mm, "acc_CI95_low": al, "acc_CI95_high": ah, "auc_CI95_low": ul, "auc_CI95_high": uh, "best_val_auc": best}.items()}
        json.dump(vit_row, open(vit_r, "w"), indent=1)
        pd.DataFrame({"path": te_p, "y_true": te_y, "p_asd": pt}).to_csv(os.path.join(RES, "test_predictions_ViTB16.csv"), index=False)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in vit_row.items()})
    vp = pd.read_csv(os.path.join(RES, "test_predictions_ViTB16.csv")).sort_values("path")
    rp = pd.read_csv(os.path.join(RES, f"test_predictions_VGG16refined_seed{XAI_SEED}.csv")).sort_values("path")
    print("DeLong p (ViT vs refined VGG16 seed 42):", round(delong_p(vp.y_true.values, vp.p_asd.values, rp.p_asd.values), 4))

    # attention rollout maps, class-averaged over the test set
    attn = []
    for blk in vit.blocks:
        blk.attn.fused_attn = False
        blk.attn.attn_drop.register_forward_hook(lambda m, i, o: attn.append(o.detach()))
    def rollout(x):
        attn.clear(); vit.eval()
        with torch.no_grad(): vit(x)
        R = torch.eye(attn[0].shape[-1], device=x.device)[None]
        for a in attn:
            a = a.float().mean(1); a = a + torch.eye(a.shape[-1], device=x.device)[None]; a = a / a.sum(-1, keepdim=True); R = a @ R
        m = R[:, 0, 1:].reshape(-1, 14, 14).cpu().numpy()
        return np.stack([np.asarray(Image.fromarray((mm / mm.max() * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255. for mm in m])
    maps = {1: [], 0: []}
    dl = DataLoader(DS(te_p, te_y, tf_eval), batch_size=32)
    for x, y in dl:
        r = rollout(x.to(dev))
        for mm, yy in zip(r, y.numpy()): maps[int(yy)].append(mm)
    a1, a0 = np.mean(maps[1], 0), np.mean(maps[0], 0)
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    ax[0].imshow(a1, cmap="jet"); ax[0].set_title("ViT attention: ASD (test)")
    ax[1].imshow(a0, cmap="jet"); ax[1].set_title("ViT attention: Non-ASD (test)")
    im = ax[2].imshow(a1 - a0, cmap="bwr"); ax[2].set_title("Difference (ASD − Non-ASD)"); plt.colorbar(im, ax=ax[2], fraction=0.046)
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.savefig(os.path.join(FIG, "vit_attention_class_average.png"), dpi=200); plt.show()

In [ ]:
# ViT occlusion test: face only vs periphery only (same masks as the VGG16 occlusion test)
if RUN_VIT:
    yy, xx = np.mgrid[0:IMG_SIZE, 0:IMG_SIZE]; c = (IMG_SIZE - 1) / 2
    face = (((xx - c) / (0.40 * IMG_SIZE)) ** 2 + ((yy - c) / (0.48 * IMG_SIZE)) ** 2) <= 1.0
    grey = np.array([124, 116, 104], np.float32)
    Xr = np.stack([load_raw(p) for p in te_p])
    conds = {"Original": Xr, "Face only": np.where(face[None, :, :, None], Xr, grey),
             "Periphery only": np.where(face[None, :, :, None], grey, Xr)}
    attn.clear(); torch.cuda.empty_cache(); vit.eval(); occ = []
    for name, arr in conds.items():
        ps = []
        for k in range(0, len(arr), 32):
            x = torch.tensor(arr[k:k + 32] / 255.0, dtype=torch.float32).permute(0, 3, 1, 2)
            x = ((x - 0.5) / 0.5).to(dev)
            with torch.no_grad(), torch.autocast(dev, enabled=dev == "cuda"):
                ps.append(torch.sigmoid(vit(x).float().squeeze(1)).cpu().numpy())
            attn.clear()   # the attention hooks store maps on every forward pass; discard them
        ps = np.concatenate(ps)
        occ.append({"model": "ViT-B/16", "condition": name, "auc": roc_auc_score(te_y, ps), "mean_P_ASD": float(ps.mean())})
    vit_occ = pd.DataFrame(occ); vit_occ.to_csv(os.path.join(RES, "vit_occlusion.csv"), index=False)
    print(vit_occ.round(4).to_string(index=False))

## Summary — send this output and the `original_sections` + `figures_original_sections` folders

In [ ]:
print("=== PART A: baseline vs refined VGG16 (mean over seeds, %) ==="); print((comp * 100).round(2).to_string())
print("DeLong p refined vs baseline per seed:", np.round(pvals, 4))
print("\n=== PART B: ERAS (n=%d) ===" % len(eras)); print(eras[["eras_gradcam", "eras_gradcampp", "uniform_baseline"]].agg(["mean", "std"]).round(4).to_string())
print(f"Wilcoxon Grad-CAM++ vs Grad-CAM p = {w.pvalue:.2e}")
print("\n=== PART C: LIME ==="); print(lime_df.drop(columns="path").agg(["mean", "std"]).round(4).to_string())
print("\n=== PART D: landmarks ==="); print(lt.round(4).to_string(index=False))
if RUN_VIT:
    print("\n=== PART E: ViT-B/16 ==="); print({k: round(v, 4) if isinstance(v, float) else v for k, v in vit_row.items()})
    print("\n=== ViT occlusion ==="); print(vit_occ.round(4).to_string(index=False))